# Part 2：
# ORL数据集人脸图像聚类

## 介绍
在本部分中，你将进行人脸图像聚类。使用的数据集为ORL数据集。该数据集由40个人，每人10张人脸图像，共400张图像构成。

请首先将Data文件夹内的ORL_jpg.zip解压缩，这里保存的是本次实验的数据。

你的任务是将这400张图像聚类为40类，并且和正确的标签对比，计算正确率。

我会给你提供一个baseline的算法，基于PCA降维和kmeans聚类。但是正确率不高，仅有62%左右。

你的任务是发挥你的聪明才智，用任何可行的手段，尽量提高聚类正确率，你的正确率将是你本次实验的成绩。

在之前ml_course环境基础上，你可能需要安装一些所需要的包：

pip install glob2

pip install opencv-python

另外，你可以安装所有你需要的包。

注意！！！

聚类是一种无监督学习的手段，在训练过程中，禁止使用labe，即真实y的值。例如，你不可以使用有监督学习的降维手段，如Linear Discriminant Analysis。

In [1]:
'''
对ORL人脸数据集进行K-means聚类
'''
import os
import numpy as np
from glob import glob
import cv2
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

下面是读取数据集的函数，读取图像并且重构成一个数据矩阵，每一行是一个样本。

In [2]:
def load_orl_dataset(data_dir):
    """
    从指定目录加载ORL数据集。

    参数
    ----------
    data_dir : str
        包含ORL数据集图片的目录。

    返回
    -------
    X : np.ndarray
        数据集，每一行为一个拉平成的一张图片。
    y : np.ndarray
        每张图片对应的标签。
    """
    image_files = glob(os.path.join(data_dir, "*", "*.jpg"))
    images = []
    labels = []

    for file in image_files:
        img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            images.append(img.flatten())
            label = os.path.basename(os.path.dirname(file))
            label = label.lstrip("s")  # 去除标签中的's'
            label = int(label)
            labels.append(int(label))

    X = np.array(images)
    y = np.array(labels)

    return X, y

下面是计算聚类正确率的函数：

In [3]:
def calculate_clustering_accuracy(y_true, y_pred):
    """
    通过比较真实标签和聚类预测标签，计算聚类准确率。

    参数
    ----------
    y_true : np.ndarray
        真实标签。
    y_pred : np.ndarray
        聚类预测标签。

    返回
    -------
    accuracy : float
        聚类准确率（百分比）。
    """
    from sklearn.metrics import adjusted_rand_score
    return adjusted_rand_score(y_true, y_pred) * 100

In [4]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def plot_tsne(X, labels, title="t-SNE visualization"):
    tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
    X_tsne = tsne.fit_transform(X)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, cmap='tab20', s=10)
    plt.title(title)
    plt.colorbar(scatter)
    plt.show()


下面是我提供的baseline方法。首先对图像进行PCA降维，然后使用kmeans聚类。

In [5]:
from sklearn.cluster import KMeans, SpectralClustering

def run_baseline():
    data_dir = r"Data\ORL_jpg"
    X, y = load_orl_dataset(data_dir)

    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    # 降维
    pca = PCA(n_components=50, svd_solver='randomized', random_state=41)
    X_reduced = pca.fit_transform(X)

    # 归一化
    from sklearn.preprocessing import normalize
    X_normalized = normalize(X_reduced)

    # 谱聚类
    spectral = SpectralClustering(
        n_clusters=40, 
        assign_labels='discretize',  # 'kmeans'
        affinity='nearest_neighbors',   # 'rbf'
        n_neighbors=7,
        # random_state=41
    )
    y_pred = spectral.fit_predict(X_normalized)


    # K-means聚类
    # kmeans = KMeans(n_clusters=40, n_init=20)
    # y_kmeans = kmeans.fit_predict(X_normalized)

    # plot_tsne(X_reduced, y_pred, title="Spectral Clustering t-SNE (on PCA features)")

    # 计算聚类准确率
    accuracy = calculate_clustering_accuracy(y, y_pred)
    print(f"Clustering Accuracy: {accuracy:.2f}%")
    

In [6]:
# 运行基线聚类
# run_baseline()

使用 FaceNet 提取人脸特征，然后使用 K-means 聚类。

In [ ]:
import torch
from facenet_pytorch import InceptionResnetV1, MTCNN
from torchvision import transforms
from PIL import Image

def run_facenet_clustering():
    data_dir = r"Data\ORL_jpg"
    X, y = load_orl_dataset(data_dir)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 加载模型
    mtcnn = MTCNN(image_size=160, margin=20, device=device)
    facenet = InceptionResnetV1(pretrained='vggface2').eval().to(device) 

    # 提取 embedding
    embeddings = []
    for i in range(len(X)):
        # 恢复 112x92 格式
        img_flat = X[i]
        img = img_flat.reshape(112, 92).astype(np.uint8)

        # 转成三通道
        pil_img = Image.fromarray(img).convert("RGB")

        # MTCNN 人脸检测+截取
        aligned = mtcnn(pil_img)
        if aligned is None:
            aligned = transforms.Resize((160, 160))(pil_img)
            aligned = transforms.ToTensor()(aligned)
        aligned = aligned.unsqueeze(0).to(device)

        # FaceNet 特征提取
        with torch.no_grad():
            emb = facenet(aligned).cpu().numpy().flatten()
        embeddings.append(emb)

    embeddings = np.array(embeddings)

    # L2 归一化
    from sklearn.preprocessing import normalize
    embeddings_norm = normalize(embeddings)

    # K-means聚类
    kmeans = KMeans(n_clusters=40, n_init=20, random_state=42)
    y_pred = kmeans.fit_predict(embeddings_norm)

    # 计算聚类准确率
    acc = calculate_clustering_accuracy(y, y_pred)
    print(f"聚类准确率: {acc:.2f}%")

    # 可视化
    # plot_tsne(embeddings_norm, y_pred, title="FaceNet + KMeans t-SNE")

run_facenet_clustering()

聚类准确率: 100.00%
